## Validation Datasets

The goal is to compare station data to ERA5 data and buoy data to GHRSST data to see how much error we are getting in a sample of regions in our models

From there, we can use that to figure out a reasonable "perturbation" for our model, changing 2m Temperature and SST accordingly to account for likely measurement error and use that to run ensemble models.

In [1]:
#import what you need
import pandas as pd #for buoy data
import re
import xarray as xr
import numpy as np
import pandas as pd
import os
from datetime import datetime, timedelta #to handle the dates
import statistics #to get mean and sd

GHRSST Data

In [2]:
#Define a function to add the right time coordinate
def preprocess(ds, filename=None):
    import re
    import pandas as pd
    from xarray import Dataset

    # Extract timestamp from filename like 20170323120000
    match = re.match(r'(\d{14})', os.path.basename(filename))
    if not match:
        raise ValueError(f"Could not extract timestamp from filename: {filename}")
    
    timestamp = pd.to_datetime(match.group(1), format='%Y%m%d%H%M%S')

    # Replace the existing time coordinate with correct timestamp
    ds = ds.assign_coords(time=("time", [timestamp]))
    return ds



In [3]:
folder = '../../01Data/GHRSST/'

file_list = sorted([
    os.path.join(folder, f)
    for f in os.listdir(folder)
    if f.endswith('.nc')
])

datasets = []
for file in file_list:
    ds = xr.open_dataset(file, decode_cf=True, mask_and_scale=True)
    ds = preprocess(ds, filename=file)
    datasets.append(ds)

combined = xr.concat(datasets, dim="time")




Get Buoy Data

In [4]:
##create useful function for reading the buoy data
def load_txt_file(filepath):
    # Step 1: Read lines
    with open(filepath, 'r') as f:
        lines = f.readlines()

    # Step 2: Find the first header line (starting with #)
    header_line = next(line for line in lines if line.startswith("#"))
    column_names = header_line.strip().lstrip("#").split()

    # Step 3: Read the data, skipping any header lines
    df = pd.read_csv(filepath, sep='\s+', comment='#', names=column_names, skiprows=2)

    # Step 4: Make sure date/time columns exist in the DataFrame
    date_cols = ['YY', 'MM', 'DD', 'hh', 'mm']
    if all(col in df.columns for col in date_cols):
        df['datetime'] = pd.to_datetime(df[date_cols].rename(
            columns={'YY': 'year', 'MM': 'month', 'DD': 'day', 'hh': 'hour', 'mm': 'minute'}
        ))
        df.drop(date_cols, axis=1, inplace=True)
        df.set_index('datetime', inplace=True)
    else:
        raise ValueError(f"Date/time columns missing in {filepath}")

    return df

In [5]:
##Buoy data

############42019
file_path = "../../01Data/ValidationDatasets/Buoys/42019h2017.txt"
df42019 = load_txt_file(file_path)

############42035
file_path = "../../01Data/ValidationDatasets/Buoys/42035h2017.txt"
df42035 = load_txt_file(file_path)

############42043
# Step 1: Read and prepare headers
file_path = "../../01Data/ValidationDatasets/Buoys/42043h2017.txt"
df42043 = load_txt_file(file_path)

############mgpt2
# Step 1: Read and prepare headers
file_path = "../../01Data/ValidationDatasets/Buoys/mgpt2h2017.txt"
dfmgpt2 = load_txt_file(file_path)

############sgnt2
# Step 1: Read and prepare headers
file_path = "../../01Data/ValidationDatasets/Buoys/sgnt2h2017.txt"
dfsgnt2 = load_txt_file(file_path)

#########Geographic Locations for the buoys
#42019, 42035, 42043, mgpt2, sgnt2
lats = [27.908, 29.235, 28.982, 29.682, 28.771]
lons = [-95.343, -94.41, -94.899, -94.985, -95.617]

#want WTMP, dates are in a column called datetime I think


From previous, atmospheric code, for context

Going to go through the buoys in loose order as listed above
* 42019
* 42035
* 42043
* mgpt2
* sgnt2

Because of how the data are set up, it is a toss up between iterating through GHRSST data versus iterating through the stations. 

In [6]:
## Get useful station information for later work
#tolerance of difference in lat/lon from target to what we use for the mean. 
lonTol = 0.1 #unlikely to get an exact match
latTol = 0.1 #unlikely to get an exact match

In [7]:
##subset your GHRSST dataset to match the different station locations
# Find subset within the tolerance range

SubsetGHRSST_42019 = combined.analysed_sst.sel(
    lat=slice(lats[0] - latTol, lats[0] + latTol),
    lon=slice(lons[0] - lonTol, lons[0] + lonTol)
)

SubsetGHRSST_42035 = combined.analysed_sst.sel(
    lat=slice(lats[1] - latTol, lats[1] + latTol),
    lon=slice(lons[1] - lonTol, lons[1] + lonTol)
)

SubsetGHRSST_42043 = combined.analysed_sst.sel(
    lat=slice(lats[2] - latTol, lats[2] + latTol),
    lon=slice(lons[2] - lonTol, lons[2] + lonTol)
)

SubsetGHRSST_mgpt2 = combined.analysed_sst.sel(
    lat=slice(lats[3] - latTol, lats[3] + latTol),
    lon=slice(lons[3] - lonTol, lons[3] + lonTol)
)

SubsetGHRSST_sgnt2 = combined.analysed_sst.sel(
    lat=slice(lats[4] - latTol, lats[4] + latTol),
    lon=slice(lons[4] - lonTol, lons[4] + lonTol)
)

In [8]:
############42019
# Store average temperatures
avg_temps = []
tolerance= timedelta(hours=12) #basically the 12 hours before and after

# Loop through each time point in SubsetGHRSST_42019
for t in SubsetGHRSST_42019.time.values:
    focal_time = pd.to_datetime(t)
    
    # Create a time window
    start_time = focal_time - tolerance
    end_time = focal_time + tolerance

    # Subset the dataframe within the time window
    matching_entries = df42019.loc[
        (df42019.index >= start_time) & 
        (df42019.index <= end_time) & 
        (df42019['WTMP'] != 999.0),
        'WTMP'
    ]

    # Calculate average if there are any matching values
    if not matching_entries.empty:
        avg = matching_entries.mean()
    else:
        avg = np.nan

    avg_temps.append(avg)

#get some summary statistics
Diffs_42019 = (SubsetGHRSST_42019.mean(dim=['lat', 'lon'], skipna=True) - 272.15).values.flatten() - avg_temps
avgDiff_42019 = statistics.mean(Diffs_42019)
medDiff_42019 = statistics.median(Diffs_42019)

In [9]:
############42035
# Store average temperatures
avg_temps = []
tolerance= timedelta(hours=12) #basically the 12 hours before and after

# Loop through each time point in SubsetGHRSST_42035
for t in SubsetGHRSST_42035.time.values:
    focal_time = pd.to_datetime(t)
    
    # Create a time window
    start_time = focal_time - tolerance
    end_time = focal_time + tolerance

    # Subset the dataframe within the time window
    matching_entries = df42019.loc[
        (df42019.index >= start_time) & 
        (df42019.index <= end_time) & 
        (df42019['WTMP'] != 999.0),
        'WTMP'
    ]

    # Calculate average if there are any matching values
    if not matching_entries.empty:
        avg = matching_entries.mean()
    else:
        avg = np.nan

    avg_temps.append(avg)

#get some summary statistics
Diffs_42035 = (SubsetGHRSST_42035.mean(dim=['lat', 'lon'], skipna=True) - 272.15).values.flatten() - avg_temps
avgDiff_42035 = statistics.mean(Diffs_42035)
medDiff_42035 = statistics.median(Diffs_42035)

In [10]:
############42043
# Store average temperatures
avg_temps = []
tolerance= timedelta(hours=12) #basically the 12 hours before and after

# Loop through each time point in SubsetGHRSST_42043
for t in SubsetGHRSST_42043.time.values:
    focal_time = pd.to_datetime(t)
    
    # Create a time window
    start_time = focal_time - tolerance
    end_time = focal_time + tolerance

    # Subset the dataframe within the time window
    matching_entries = df42019.loc[
        (df42019.index >= start_time) & 
        (df42019.index <= end_time) & 
        (df42019['WTMP'] != 999.0),
        'WTMP'
    ]

    # Calculate average if there are any matching values
    if not matching_entries.empty:
        avg = matching_entries.mean()
    else:
        avg = np.nan

    avg_temps.append(avg)

#get some summary statistics
Diffs_42043 = (SubsetGHRSST_42043.mean(dim=['lat', 'lon'], skipna=True) - 272.15).values.flatten() - avg_temps
avgDiff_42043 = statistics.mean(Diffs_42043)
medDiff_42043 = statistics.median(Diffs_42043)

In [11]:
############mgpt2
# Store average temperatures
avg_temps = []
tolerance= timedelta(hours=12) #basically the 12 hours before and after

# Loop through each time point in SubsetGHRSST_mgpt2
for t in SubsetGHRSST_mgpt2.time.values:
    focal_time = pd.to_datetime(t)
    
    # Create a time window
    start_time = focal_time - tolerance
    end_time = focal_time + tolerance

    # Subset the dataframe within the time window
    matching_entries = df42019.loc[
        (df42019.index >= start_time) & 
        (df42019.index <= end_time) & 
        (df42019['WTMP'] != 999.0),
        'WTMP'
    ]

    # Calculate average if there are any matching values
    if not matching_entries.empty:
        avg = matching_entries.mean()
    else:
        avg = np.nan

    avg_temps.append(avg)

#get some summary statistics
Diffs_mgpt2 = (SubsetGHRSST_mgpt2.mean(dim=['lat', 'lon'], skipna=True) - 272.15).values.flatten() - avg_temps
avgDiff_mgpt2 = statistics.mean(Diffs_mgpt2)
medDiff_mgpt2 = statistics.median(Diffs_mgpt2)

In [12]:
############sgnt2
# Store average temperatures
avg_temps = []
tolerance= timedelta(hours=12) #basically the 12 hours before and after

# Loop through each time point in SubsetGHRSST_sgnt2
for t in SubsetGHRSST_sgnt2.time.values:
    focal_time = pd.to_datetime(t)
    
    # Create a time window
    start_time = focal_time - tolerance
    end_time = focal_time + tolerance

    # Subset the dataframe within the time window
    matching_entries = df42019.loc[
        (df42019.index >= start_time) & 
        (df42019.index <= end_time) & 
        (df42019['WTMP'] != 999.0),
        'WTMP'
    ]

    # Calculate average if there are any matching values
    if not matching_entries.empty:
        avg = matching_entries.mean()
    else:
        avg = np.nan

    avg_temps.append(avg)

#get some summary statistics
Diffs_sgnt2 = (SubsetGHRSST_sgnt2.mean(dim=['lat', 'lon'], skipna=True) - 272.15).values.flatten() - avg_temps
avgDiff_sgnt2 = statistics.mean(Diffs_sgnt2)
medDiff_sgnt2 = statistics.median(Diffs_sgnt2)

Summarize into output file

In [13]:
means = [avgDiff_42019, avgDiff_42035, avgDiff_42043, avgDiff_mgpt2, avgDiff_sgnt2]
medians = [medDiff_42019, medDiff_42035, medDiff_42043, medDiff_mgpt2, medDiff_sgnt2]
OutputDF = pd.DataFrame(np.column_stack((means, medians)), columns=["mean", "median"], 
                        index=["Buoy42019", "Buoy42035", "Buoy42043", "Buoymgpt2", "Buoysgnt2"])
OutputDF.to_csv("../../03ProcessedData/GHRSSTErrors.csv")

In [14]:
#Instead organize it so we can see more of what these errors look like
days=['Mar23', 'Mar24', 'Mar25', 'Mar26', 'Mar27', 'Mar28', 'Mar29', 'Mar30', 'Mar31', 'Apr01', 'Apr02', 'Apr03', 'Apr04']
OutputDF = pd.DataFrame(np.column_stack((days, Diffs_42019, Diffs_42035, Diffs_42043, Diffs_mgpt2, Diffs_sgnt2)), 
                        columns=["Days", "42019", "42035", "42043", "mgpt2", "sgnt2"])

OutputDF.to_csv("../../03ProcessedData/GHRSSTErrors_byday.csv")
